In [0]:
# Demo 11 Setup: Star Schemas vs Pre-Joined Tables
# Creates a star schema (fact + dimensions) and a pre-joined gold table
# to demonstrate query complexity, shuffling, and performance tradeoffs.

import re
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DoubleType
from datetime import date, timedelta
import random

# --- Schema setup ---
username = spark.sql("SELECT current_user()").collect()[0][0]
clean_username = re.sub(r'[^a-zA-Z0-9]', '_', username.split('@')[0])
catalog = "main"
schema = f"demo_star_schema_{clean_username}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`")
spark.sql(f"USE CATALOG `{catalog}`")
spark.sql(f"USE SCHEMA `{schema}`")

print(f"Schema: {catalog}.{schema}")

In [0]:
# --- dim_regions: 8 rows ---
spark.sql("DROP TABLE IF EXISTS dim_regions")
spark.sql("""
  CREATE TABLE dim_regions (
    region_id INT,
    region_name STRING,
    country STRING,
    timezone STRING
  )
""")
spark.sql("""
  INSERT INTO dim_regions VALUES
    (1, 'Northeast', 'USA', 'America/New_York'),
    (2, 'Southeast', 'USA', 'America/New_York'),
    (3, 'Midwest', 'USA', 'America/Chicago'),
    (4, 'Southwest', 'USA', 'America/Denver'),
    (5, 'West', 'USA', 'America/Los_Angeles'),
    (6, 'Pacific Northwest', 'USA', 'America/Los_Angeles'),
    (7, 'Central Canada', 'Canada', 'America/Toronto'),
    (8, 'Western Canada', 'Canada', 'America/Vancouver')
""")
print("Created dim_regions: 8 rows")

In [0]:
# --- dim_customers: 50 rows ---
spark.sql("DROP TABLE IF EXISTS dim_customers")
spark.sql("""
  CREATE TABLE dim_customers (
    customer_id INT,
    customer_name STRING,
    segment STRING,
    loyalty_tier STRING
  )
""")

random.seed(42)
segments = ['Enterprise', 'Mid-Market', 'SMB', 'Consumer']
tiers = ['Gold', 'Silver', 'Bronze', 'None']
first_names = ['James', 'Emma', 'Liam', 'Olivia', 'Noah', 'Ava', 'Ethan', 'Sophia', 'Mason', 'Isabella']
last_names = ['Smith', 'Johnson', 'Williams', 'Brown', 'Jones', 'Garcia', 'Miller', 'Davis', 'Wilson', 'Taylor']

insert_values = []
for i in range(1, 51):
    name = f"{random.choice(first_names)} {random.choice(last_names)}"
    segment = segments[(i - 1) % 4]
    tier = tiers[(i - 1) % 4]
    insert_values.append(f"({i}, '{name}', '{segment}', '{tier}')")

spark.sql(f"INSERT INTO dim_customers VALUES {', '.join(insert_values)}")
print("Created dim_customers: 50 rows")

In [0]:
# --- dim_products: 25 rows ---
spark.sql("DROP TABLE IF EXISTS dim_products")
spark.sql("""
  CREATE TABLE dim_products (
    product_id INT,
    product_name STRING,
    category STRING,
    subcategory STRING,
    brand STRING
  )
""")

product_data = [
    (1, 'Pro Laptop 15', 'Electronics', 'Laptops', 'TechCorp'),
    (2, 'UltraBook Air', 'Electronics', 'Laptops', 'SlimTech'),
    (3, 'SmartPhone X', 'Electronics', 'Phones', 'TechCorp'),
    (4, 'Budget Phone', 'Electronics', 'Phones', 'ValueBrand'),
    (5, 'Wireless Earbuds', 'Electronics', 'Audio', 'SoundMax'),
    (6, 'Mens Jacket', 'Clothing', 'Outerwear', 'UrbanWear'),
    (7, 'Womens Blazer', 'Clothing', 'Outerwear', 'StyleCo'),
    (8, 'Running Shoes', 'Clothing', 'Footwear', 'SpeedFit'),
    (9, 'Casual Sneakers', 'Clothing', 'Footwear', 'UrbanWear'),
    (10, 'Cotton T-Shirt', 'Clothing', 'Basics', 'ValueBrand'),
    (11, 'Standing Desk', 'Home', 'Furniture', 'ErgoLife'),
    (12, 'Office Chair', 'Home', 'Furniture', 'ErgoLife'),
    (13, 'Desk Lamp', 'Home', 'Lighting', 'BrightHome'),
    (14, 'Bookshelf', 'Home', 'Furniture', 'WoodCraft'),
    (15, 'Area Rug', 'Home', 'Decor', 'CozyHome'),
    (16, 'Yoga Mat', 'Sports', 'Fitness', 'FlexGear'),
    (17, 'Dumbbells Set', 'Sports', 'Fitness', 'IronPump'),
    (18, 'Tennis Racket', 'Sports', 'Racquet', 'ProServe'),
    (19, 'Cycling Helmet', 'Sports', 'Cycling', 'SafeRide'),
    (20, 'Hiking Backpack', 'Sports', 'Outdoor', 'TrailMaster'),
    (21, 'Organic Coffee', 'Food', 'Beverages', 'FreshBrew'),
    (22, 'Protein Bars', 'Food', 'Snacks', 'FuelUp'),
    (23, 'Green Tea Box', 'Food', 'Beverages', 'ZenLeaf'),
    (24, 'Trail Mix', 'Food', 'Snacks', 'NatureBite'),
    (25, 'Sparkling Water', 'Food', 'Beverages', 'PureFizz')
]

values_str = ', '.join([f"({p[0]}, '{p[1]}', '{p[2]}', '{p[3]}', '{p[4]}')" for p in product_data])
spark.sql(f"INSERT INTO dim_products VALUES {values_str}")
print("Created dim_products: 25 rows")

In [0]:
# --- fact_sales: 3000 rows ---
spark.sql("DROP TABLE IF EXISTS fact_sales")

random.seed(123)

schema_def = StructType([
    StructField("sale_id", IntegerType(), False),
    StructField("sale_date", DateType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("product_id", IntegerType(), False),
    StructField("region_id", IntegerType(), False),
    StructField("quantity", IntegerType(), False),
    StructField("unit_price", DoubleType(), False),
    StructField("discount_pct", DoubleType(), False)
])

rows = []
for i in range(1, 3001):
    sale_date = date(2024, 1, 1) + timedelta(days=random.randint(0, 364))
    rows.append((
        i,
        sale_date,
        random.randint(1, 50),
        random.randint(1, 25),
        random.randint(1, 8),
        random.randint(1, 10),
        round(random.uniform(20, 500), 2),
        round(random.choice([0, 0, 0, 0, 0.05, 0.10, 0.15, 0.20]), 2)
    ))

df = spark.createDataFrame(rows, schema=schema_def)
df.write.saveAsTable("fact_sales")
print(f"Created fact_sales: {df.count()} rows")

In [0]:
# --- gold_sales_wide: Pre-joined wide table (the Gold layer pattern) ---
spark.sql("DROP TABLE IF EXISTS gold_sales_wide")

spark.sql("""
  CREATE TABLE gold_sales_wide AS
  SELECT
    f.sale_id,
    f.sale_date,
    YEAR(f.sale_date) AS sale_year,
    QUARTER(f.sale_date) AS sale_quarter,
    MONTH(f.sale_date) AS sale_month,
    c.customer_name,
    c.segment AS customer_segment,
    c.loyalty_tier,
    p.product_name,
    p.category AS product_category,
    p.subcategory AS product_subcategory,
    p.brand,
    r.region_name,
    r.country,
    f.quantity,
    f.unit_price,
    f.discount_pct,
    ROUND(f.quantity * f.unit_price * (1 - f.discount_pct), 2) AS net_revenue
  FROM fact_sales f
  INNER JOIN dim_customers c ON f.customer_id = c.customer_id
  INNER JOIN dim_products p ON f.product_id = p.product_id
  INNER JOIN dim_regions r ON f.region_id = r.region_id
""")

gold_count = spark.sql("SELECT COUNT(*) FROM gold_sales_wide").collect()[0][0]
print(f"Created gold_sales_wide: {gold_count} rows (pre-joined with all dimensions)")

In [0]:
# --- Summary ---
print("="*60)
print("SETUP COMPLETE")
print("="*60)
print(f"Catalog:  {catalog}")
print(f"Schema:   {schema}")
print(f"")
print(f"Star Schema (Silver layer):")
print(f"  fact_sales     - 3000 rows (sale_id, sale_date, customer_id, product_id, region_id, quantity, unit_price, discount_pct)")
print(f"  dim_customers  - 50 rows")
print(f"  dim_products   - 25 rows")
print(f"  dim_regions    - 8 rows")
print(f"")
print(f"Pre-joined Gold table:")
print(f"  gold_sales_wide - {gold_count} rows (all dimensions joined, net_revenue pre-calculated)")
print(f"")
print(f"Next: Open the SQL Editor and run:")
print(f"  USE CATALOG {catalog};")
print(f"  USE SCHEMA {schema};")
print(f"")
print(f"Then follow the Demo 11 notebook.")